## Welcome home, captain

In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F
tables = ["mta_hourly_2017_2019", "mta_hourly_2020_2024", "mta_hourly_2025"]
for t in tables:
    d = spark.table(f"bronze.{t}")
    print(t, d.count())
    print(d.columns)
    d.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in d.columns]).show(truncate=False)

StatementMeta(, 8cb5a019-6811-426f-b5b6-fddd925fd89c, 3, Finished, Available, Finished, False)

mta_hourly_2017_2019 57611916
['transit_timestamp', 'transit_mode', 'station_complex_id', 'station_complex', 'borough', 'payment_method', 'fare_class_category', 'ridership', 'transfers', 'latitude', 'longitude', 'Georeference', 'source_dataset', 'load_date']
+-----------------+------------+------------------+---------------+-------+--------------+-------------------+---------+---------+--------+---------+------------+--------------+---------+
|transit_timestamp|transit_mode|station_complex_id|station_complex|borough|payment_method|fare_class_category|ridership|transfers|latitude|longitude|Georeference|source_dataset|load_date|
+-----------------+------------+------------------+---------------+-------+--------------+-------------------+---------+---------+--------+---------+------------+--------------+---------+
|0                |0           |0                 |0              |0      |0             |0                  |0        |0        |0       |0        |0           |0             |

In [1]:
from pyspark.sql import functions as F
names = ["mta_hourly_2017_2019", "mta_hourly_2020_2024", "mta_hourly_2025"]
b = None
for n in names:
    d = spark.table(f"bronze.{n}")
    b = d if b is None else b.unionByName(d)
b.createOrReplaceTempView("bronze_all")
b.groupBy("source_dataset").count().show(truncate=False)

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 3, Finished, Available, Finished, False)

+--------------+---------+
|source_dataset|count    |
+--------------+---------+
|t69i-h2me     |57611916 |
|wujg-7c2s     |120855567|
|5wq4-mkjj     |45962558 |
+--------------+---------+



In [2]:
spark.sql("""
SELECT source_dataset, transit_mode, borough, payment_method, fare_class_category, COUNT(*) AS n
FROM bronze_all
GROUP BY 1,2,3,4,5
ORDER BY 1,2,3,4,5
""").show(300, truncate=False)

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 4, Finished, Available, Finished, False)

+--------------+---------------------+-------------+--------------+--------------------------------+-------+
|source_dataset|transit_mode         |borough      |payment_method|fare_class_category             |n      |
+--------------+---------------------+-------------+--------------+--------------------------------+-------+
|5wq4-mkjj     |staten_island_railway|Staten Island|metrocard     |Metrocard - Fair Fare           |9462   |
|5wq4-mkjj     |staten_island_railway|Staten Island|metrocard     |Metrocard - Full Fare           |14695  |
|5wq4-mkjj     |staten_island_railway|Staten Island|metrocard     |Metrocard - Other               |19761  |
|5wq4-mkjj     |staten_island_railway|Staten Island|metrocard     |Metrocard - Seniors & Disability|9164   |
|5wq4-mkjj     |staten_island_railway|Staten Island|metrocard     |Metrocard - Unlimited 30-Day    |11718  |
|5wq4-mkjj     |staten_island_railway|Staten Island|metrocard     |Metrocard - Unlimited 7-Day     |12445  |
|5wq4-mkjj     |sta

In [3]:
t = spark.sql("SELECT source_dataset, transit_timestamp FROM bronze_all") \
         .withColumn("ts", F.to_timestamp("transit_timestamp", "MM/dd/yyyy hh:mm:ss a"))
t.groupBy("source_dataset").agg(
    F.count("*").alias("rows"),
    F.sum(F.col("ts").isNull().cast("int")).alias("unparsed"),
    F.min("ts").alias("first_ts"),
    F.max("ts").alias("last_ts")).show(truncate=False)

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 5, Finished, Available, Finished, False)

+--------------+---------+--------+-------------------+-------------------+
|source_dataset|rows     |unparsed|first_ts           |last_ts            |
+--------------+---------+--------+-------------------+-------------------+
|t69i-h2me     |57611916 |0       |2017-01-13 00:00:00|2019-12-31 23:00:00|
|wujg-7c2s     |120855567|0       |2020-01-01 00:00:00|2024-12-31 23:00:00|
|5wq4-mkjj     |45962558 |0       |2025-01-01 00:00:00|2026-09-30 23:00:00|
+--------------+---------+--------+-------------------+-------------------+



In [4]:
spark.sql("""
SELECT station_complex_id,
       COUNT(DISTINCT station_complex) AS names,
       COUNT(DISTINCT latitude, longitude) AS coords
FROM bronze_all
GROUP BY 1
HAVING COUNT(DISTINCT station_complex) > 1 OR COUNT(DISTINCT latitude, longitude) > 1
ORDER BY 1
""").show(100, truncate=False)

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 6, Finished, Available, Finished, False)

+------------------+-----+------+
|station_complex_id|names|coords|
+------------------+-----+------+
|221               |2    |1     |
|222               |2    |1     |
|223               |2    |1     |
|224               |2    |1     |
|273               |2    |1     |
|276               |2    |1     |
|323               |2    |1     |
|405               |2    |1     |
|441               |2    |1     |
|601               |2    |5     |
|602               |2    |4     |
|603               |1    |2     |
|604               |2    |4     |
|605               |2    |3     |
|606               |2    |7     |
|607               |1    |3     |
|608               |2    |3     |
|609               |2    |4     |
|610               |2    |2     |
|611               |2    |7     |
|612               |2    |3     |
|613               |2    |3     |
|614               |2    |3     |
|615               |2    |4     |
|616               |2    |4     |
|617               |2    |4     |
|618          

In [5]:
spark.sql("""
SELECT SUM(CASE WHEN TRY_CAST(ridership AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS bad_ridership,
       SUM(CASE WHEN TRY_CAST(transfers AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS bad_transfers,
       SUM(CASE WHEN CAST(ridership AS DOUBLE) <> FLOOR(CAST(ridership AS DOUBLE)) THEN 1 ELSE 0 END) AS fractional_ridership
FROM bronze_all
""").show()

spark.sql("""
SELECT COUNT(*) AS duplicate_keys FROM (
  SELECT 1 FROM bronze_all
  GROUP BY transit_timestamp, station_complex_id, payment_method, fare_class_category
  HAVING COUNT(*) > 1)
""").show()

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 7, Finished, Available, Finished, False)

+-------------+-------------+--------------------+
|bad_ridership|bad_transfers|fractional_ridership|
+-------------+-------------+--------------------+
|            0|            0|                   0|
+-------------+-------------+--------------------+

+--------------+
|duplicate_keys|
+--------------+
|             0|
+--------------+



In [6]:
spark.sql("""
SELECT station_complex_id,
       MAX(CAST(latitude AS DOUBLE)) - MIN(CAST(latitude AS DOUBLE)) AS lat_range,
       MAX(CAST(longitude AS DOUBLE)) - MIN(CAST(longitude AS DOUBLE)) AS lon_range
FROM bronze_all
GROUP BY 1
HAVING COUNT(DISTINCT latitude, longitude) > 1
ORDER BY lat_range DESC
""").show(50, truncate=False)

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 8, Finished, Available, Finished, False)

+------------------+---------------------+---------------------+
|station_complex_id|lat_range            |lon_range            |
+------------------+---------------------+---------------------+
|624               |0.0034429999999971983|0.0024500000000102773|
|611               |0.0026370000000000005|0.0029849999999953525|
|620               |0.0016940000000005284|0.0017760000000066611|
|615               |0.0016279999999966321|5.449999999882493E-4 |
|623               |0.0014359999999982165|0.001890000000003056 |
|606               |0.0012899999999973488|0.0021699999999924557|
|629               |0.0012700000000052114|0.0011459999999914317|
|618               |0.0011170000000007008|8.899999999982811E-4 |
|602               |0.0010639999999995098|6.200000000120554E-4 |
|635               |0.0010169999999973811|6.740000000036162E-4 |
|628               |9.579999999971278E-4 |0.0029400000000094906|
|601               |8.929999999978122E-4 |0.003990000000001714 |
|617               |7.939

In [7]:
spark.sql("""
SELECT station_complex_id, source_dataset, station_complex, borough, latitude, longitude,
       COUNT(*) AS n,
       MIN(to_timestamp(transit_timestamp, 'MM/dd/yyyy hh:mm:ss a')) AS first_ts,
       MAX(to_timestamp(transit_timestamp, 'MM/dd/yyyy hh:mm:ss a')) AS last_ts
FROM bronze_all
WHERE station_complex_id IN ('221', '601', '628', '603')
GROUP BY 1,2,3,4,5,6
ORDER BY 1,2,3,5,6
""").show(200, truncate=False)

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 9, Finished, Available, Finished, False)

+------------------+--------------+------------------------------+---------+---------+----------+------+-------------------+-------------------+
|station_complex_id|source_dataset|station_complex               |borough  |latitude |longitude |n     |first_ts           |last_ts            |
+------------------+--------------+------------------------------+---------+---------+----------+------+-------------------+-------------------+
|221               |5wq4-mkjj     |21 St-Queensbridge (M)        |Queens   |40.754204|-73.94283 |123154|2025-01-01 00:00:00|2026-09-30 23:00:00|
|221               |t69i-h2me     |21 St-Queensbridge (F)        |Queens   |40.754204|-73.94283 |144438|2017-01-13 00:00:00|2019-12-31 23:00:00|
|221               |wujg-7c2s     |21 St-Queensbridge (F)        |Queens   |40.754204|-73.94283 |292181|2020-01-01 00:00:00|2024-12-31 23:00:00|
|601               |5wq4-mkjj     |14 St/6 Av (1,2,3,L,F,M)      |Manhattan|40.737797|-73.997734|132831|2025-01-01 00:00:00|2026-0

In [8]:
for n in ["mta_hourly_2017_2019", "mta_hourly_2020_2024", "mta_hourly_2025"]:
    spark.table(f"bronze.{n}").select("source_dataset", "load_date").limit(2).show(truncate=False)

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 10, Finished, Available, Finished, False)

+--------------+----------------------------+
|source_dataset|load_date                   |
+--------------+----------------------------+
|t69i-h2me     |2026-10-10T01:26:24.9718315Z|
|t69i-h2me     |2026-10-10T01:26:24.9718315Z|
+--------------+----------------------------+

+--------------+----------------------------+
|source_dataset|load_date                   |
+--------------+----------------------------+
|wujg-7c2s     |2026-10-10T01:14:13.8032224Z|
|wujg-7c2s     |2026-10-10T01:14:13.8032224Z|
+--------------+----------------------------+

+--------------+----------------------------+
|source_dataset|load_date                   |
+--------------+----------------------------+
|5wq4-mkjj     |2026-10-10T00:13:46.2197596Z|
|5wq4-mkjj     |2026-10-10T00:13:46.2197596Z|
+--------------+----------------------------+



## Building Silver Layer

In [9]:
from pyspark.sql import functions as F

names = ["mta_hourly_2017_2019", "mta_hourly_2020_2024", "mta_hourly_2025"]
bronze = None
for n in names:
    d = spark.table(f"bronze.{n}")
    bronze = d if bronze is None else bronze.unionByName(d)

silver = (bronze
    .withColumn("transit_ts", F.to_timestamp("transit_timestamp", "MM/dd/yyyy hh:mm:ss a"))
    .withColumn("ridership_date", F.to_date("transit_ts"))
    .withColumn("ridership_year", F.year("transit_ts"))
    .withColumn("hour_of_day", F.hour("transit_ts"))
    .withColumn("transit_mode",
        F.when(F.col("transit_mode") == "subway", "Subway")
         .when(F.col("transit_mode") == "staten_island_railway", "Staten Island Railway")
         .when(F.col("transit_mode") == "tram", "Roosevelt Island Tram")
         .otherwise(F.col("transit_mode")))
    .withColumn("payment_method",
        F.when(F.col("payment_method") == "omny", "OMNY")
         .when(F.col("payment_method") == "metrocard", "MetroCard")
         .otherwise(F.col("payment_method")))
    .withColumn("fare_class_category",
        F.regexp_replace("fare_class_category", "^Metrocard", "MetroCard"))
    .withColumn("ridership", F.col("ridership").cast("double").cast("long"))
    .withColumn("transfers", F.col("transfers").cast("double").cast("long"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("load_date", F.to_timestamp(F.substring("load_date", 1, 19)))
    .select("transit_ts", "ridership_date", "ridership_year", "hour_of_day",
            "transit_mode", "station_complex_id", "station_complex", "borough",
            "payment_method", "fare_class_category", "ridership", "transfers",
            "latitude", "longitude", "source_dataset", "load_date"))

(silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ridership_year")
    .saveAsTable("silver.ridership_hourly"))

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 11, Finished, Available, Finished, False)

In [10]:
from pyspark.sql import functions as F

s = spark.table("silver.ridership_hourly")
b = None
for n in ["mta_hourly_2017_2019", "mta_hourly_2020_2024", "mta_hourly_2025"]:
    d = spark.table(f"bronze.{n}")
    b = d if b is None else b.unionByName(d)

print("SILVER")
s.agg(F.count("*").alias("rows"),
      F.sum("ridership").alias("total_ridership"),
      F.sum(F.col("transit_ts").isNull().cast("int")).alias("null_ts"),
      F.min("transit_ts").alias("first_ts"),
      F.max("transit_ts").alias("last_ts")).show(truncate=False)

print("BRONZE")
b.agg(F.count("*").alias("rows"),
      F.sum(F.col("ridership").cast("double")).alias("total_ridership")).show(truncate=False)

s.select("payment_method", "fare_class_category").distinct() \
 .orderBy("fare_class_category").show(20, truncate=False)

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 12, Finished, Available, Finished, False)

SILVER
+---------+---------------+-------+-------------------+-------------------+
|rows     |total_ridership|null_ts|first_ts           |last_ts            |
+---------+---------------+-------+-------------------+-------------------+
|224430041|12155332038    |0      |2017-01-13 00:00:00|2026-09-30 23:00:00|
+---------+---------------+-------+-------------------+-------------------+

BRONZE
+---------+---------------+
|rows     |total_ridership|
+---------+---------------+
|224430041|1.2155332038E10|
+---------+---------------+

+--------------+--------------------------------+
|payment_method|fare_class_category             |
+--------------+--------------------------------+
|MetroCard     |MetroCard - Fair Fare           |
|MetroCard     |MetroCard - Full Fare           |
|MetroCard     |MetroCard - Other               |
|MetroCard     |MetroCard - Seniors & Disability|
|MetroCard     |MetroCard - Students            |
|MetroCard     |MetroCard - Unlimited 30-Day    |
|MetroCard    

In [11]:
s = spark.table("silver.ridership_hourly")
s.agg(F.min("ridership").alias("min_ridership"),
      F.max("ridership").alias("max_ridership"),
      F.min("transfers").alias("min_transfers"),
      F.sum((F.col("transfers") > F.col("ridership")).cast("int")).alias("transfers_gt_ridership"),
      F.sum((F.minute("transit_ts") != 0).cast("int")).alias("not_on_the_hour"),
      F.sum(F.col("load_date").isNull().cast("int")).alias("null_load_date")).show()
s.groupBy("transit_mode").count().show(truncate=False)

StatementMeta(, 031a5697-ea09-434f-bfa1-0b9fc9125af2, 13, Finished, Available, Finished, False)

+-------------+-------------+-------------+----------------------+---------------+--------------+
|min_ridership|max_ridership|min_transfers|transfers_gt_ridership|not_on_the_hour|null_load_date|
+-------------+-------------+-------------+----------------------+---------------+--------------+
|            1|        23510|          -30|                     0|              0|             0|
+-------------+-------------+-------------+----------------------+---------------+--------------+

+---------------------+---------+
|transit_mode         |count    |
+---------------------+---------+
|Subway               |222714492|
|Staten Island Railway|877727   |
|Roosevelt Island Tram|837822   |
+---------------------+---------+



In [1]:
from pyspark.sql import functions as F
s = spark.table("silver.ridership_hourly")
neg = s.filter("transfers < 0")
print("rows with negative transfers:", neg.count())
neg.groupBy("source_dataset", "ridership_year").count().orderBy("ridership_year").show()
neg.orderBy("transfers").select("transit_ts", "station_complex", "fare_class_category",
                                 "ridership", "transfers").show(10, truncate=False)

StatementMeta(, b1ccebb1-3f49-4833-b5e7-717d7917e407, 3, Finished, Available, Finished, False)

rows with negative transfers: 250
+--------------+--------------+-----+
|source_dataset|ridership_year|count|
+--------------+--------------+-----+
|     5wq4-mkjj|          2026|  250|
+--------------+--------------+-----+

+-------------------+------------------------+-------------------+---------+---------+
|transit_ts         |station_complex         |fare_class_category|ridership|transfers|
+-------------------+------------------------+-------------------+---------+---------+
|2026-07-26 14:00:00|14 St/6 Av (1,2,3,L,F,M)|OMNY - Other       |74       |-30      |
|2026-06-28 16:00:00|14 St/6 Av (1,2,3,L,F,M)|OMNY - Full Fare   |4819     |-5       |
|2026-08-14 11:00:00|14 St/6 Av (1,2,3,L,F,M)|OMNY - Fair Fare   |22       |-3       |
|2026-08-15 19:00:00|14 St/6 Av (1,2,3,L,F,M)|OMNY - Full Fare   |1763     |-3       |
|2026-08-09 00:00:00|14 St/6 Av (1,2,3,L,F,M)|OMNY - Full Fare   |655      |-3       |
|2026-05-26 18:00:00|14 St/6 Av (1,2,3,L,F,M)|OMNY - Fair Fare   |93       |-2 

In [1]:
from pyspark.sql import functions as F, Window

s = spark.table("silver.ridership_hourly")

# ---------- dim_station ----------
variants = (s.groupBy("station_complex_id", "source_dataset", "station_complex",
                      "borough", "transit_mode", "latitude", "longitude")
              .agg(F.count("*").alias("n"),
                   F.min("transit_ts").alias("first_seen"),
                   F.max("transit_ts").alias("last_seen")))

w_ds = Window.partitionBy("station_complex_id", "source_dataset")
w_pick = Window.partitionBy("station_complex_id") \
               .orderBy(F.col("ds_last").desc(), F.col("n").desc())

picked = (variants.withColumn("ds_last", F.max("last_seen").over(w_ds))
                  .withColumn("rn", F.row_number().over(w_pick))
                  .filter("rn = 1")
                  .select("station_complex_id", "station_complex", "borough",
                          "transit_mode", "latitude", "longitude"))

span = (variants.groupBy("station_complex_id")
                .agg(F.min("first_seen").alias("first_seen"),
                     F.max("last_seen").alias("last_seen")))

dim_station = (picked.join(span, "station_complex_id")
    .withColumn("routes", F.regexp_extract("station_complex", r"\(([^)]*)\)\s*$", 1))
    .withColumn("station_key", F.row_number().over(Window.orderBy("station_complex_id")))
    .select("station_key", "station_complex_id", "station_complex", "routes",
            "borough", "transit_mode", "latitude", "longitude", "first_seen", "last_seen"))

assert dim_station.count() == s.select("station_complex_id").distinct().count(), "one row per station ID expected"
dim_station.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_station")

# ---------- dim_fare_class ----------
dim_fare = (s.groupBy("fare_class_category", "payment_method")
    .agg(F.min("ridership_date").alias("first_seen"),
         F.max("ridership_date").alias("last_seen"))
    .withColumn("fare_group",
        F.when(F.col("fare_class_category").contains("Unlimited"), "Unlimited")
         .when(F.col("fare_class_category").contains("Seniors"), "Seniors & Disability")
         .when(F.col("fare_class_category").contains("Students"), "Students")
         .when(F.col("fare_class_category").contains("Fair Fare"), "Fair Fare")
         .when(F.col("fare_class_category").contains("Full Fare"), "Full Fare")
         .otherwise("Other"))
    .withColumn("fare_class_key", F.row_number().over(Window.orderBy("fare_class_category")))
    .select("fare_class_key", "fare_class_category", "payment_method",
            "fare_group", "first_seen", "last_seen"))

assert dim_fare.count() == 12, "expected 12 fare classes"
dim_fare.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_fare_class")

# ---------- dim_date ----------
b = s.agg(F.min("ridership_date").alias("mn"), F.max("ridership_date").alias("mx")).first()
dim_date = (spark.sql(f"""
    SELECT explode(sequence(to_date('{b.mn.year}-01-01'), to_date('{b.mx.year}-12-31'), interval 1 day)) AS date""")
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_of_week", ((F.dayofweek("date") + 5) % 7) + 1)
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))
    .select("date_key", "date", "year", "quarter", "month", "month_name", "year_month",
            "week_of_year", "day_of_month", "day_of_week", "day_name", "is_weekend"))
dim_date.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_date")

for t in ["dim_station", "dim_fare_class", "dim_date"]:
    print(t, spark.table(f"gold.{t}").count())

StatementMeta(, 3692a400-3871-4b11-8a3a-6a0b28097f82, 3, Finished, Available, Finished, False)

dim_station 428
dim_fare_class 12
dim_date 3652


In [2]:
s = spark.table("silver.ridership_hourly")
st = spark.table("gold.dim_station").select("station_key", "station_complex_id")
fc = spark.table("gold.dim_fare_class").select("fare_class_key", "fare_class_category")

daily = (s.groupBy("ridership_date", "station_complex_id", "fare_class_category")
           .agg(F.sum("ridership").alias("ridership"),
                F.sum("transfers").alias("transfers")))

hours = (s.groupBy("ridership_date", "station_complex_id")
           .agg(F.countDistinct("hour_of_day").cast("int").alias("hours_with_data")))

fact = (daily.join(hours, ["ridership_date", "station_complex_id"])
        .join(st, "station_complex_id")
        .join(fc, "fare_class_category")
        .withColumn("date_key", F.date_format("ridership_date", "yyyyMMdd").cast("int"))
        .select("date_key", "station_key", "fare_class_key",
                "ridership", "transfers", "hours_with_data"))

fact.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.fact_ridership_daily")

StatementMeta(, 3692a400-3871-4b11-8a3a-6a0b28097f82, 4, Finished, Available, Finished, False)

In [3]:
f = spark.table("gold.fact_ridership_daily")
print("FACT")
f.agg(F.count("*").alias("rows"),
      F.sum("ridership").alias("total_ridership"),
      F.sum("transfers").alias("total_transfers"),
      F.sum(F.col("station_key").isNull().cast("int")).alias("null_station"),
      F.sum(F.col("fare_class_key").isNull().cast("int")).alias("null_fare"),
      F.max("hours_with_data").alias("max_hours")).show()
print("SILVER")
spark.table("silver.ridership_hourly").agg(
      F.sum("ridership").alias("total_ridership"),
      F.sum("transfers").alias("total_transfers")).show()

StatementMeta(, 3692a400-3871-4b11-8a3a-6a0b28097f82, 5, Finished, Available, Finished, False)

FACT
+--------+---------------+---------------+------------+---------+---------+
|    rows|total_ridership|total_transfers|null_station|null_fare|max_hours|
+--------+---------------+---------------+------------+---------+---------+
|11975113|    12155332038|      482835836|           0|        0|       24|
+--------+---------------+---------------+------------+---------+---------+

SILVER
+---------------+---------------+
|total_ridership|total_transfers|
+---------------+---------------+
|    12155332038|      482835836|
+---------------+---------------+



In [4]:
%pip install holidays

StatementMeta(, 3692a400-3871-4b11-8a3a-6a0b28097f82, 10, Finished, Available, Finished, True)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /nfs4/pyenv-84b1988d-e370-4716-81d1-1d9d1f8c9d90/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [6]:
import holidays
from pyspark.sql import functions as F

us = holidays.country_holidays("US", years=range(2017, 2028))
hol = spark.createDataFrame([(d, n) for d, n in us.items()],
                            "holiday_date date, holiday_name string")

dd = spark.table("gold.dim_date").drop("is_holiday", "holiday_name")
dd = (dd.join(hol, dd["date"] == hol["holiday_date"], "left")
        .withColumn("is_holiday", F.col("holiday_date").isNotNull())
        .drop("holiday_date")
        .orderBy("date")
        .cache())
dd.count()
dd.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_date")

spark.table("gold.dim_date").filter("is_holiday").orderBy("date") \
     .select("date", "holiday_name").filter("year(date) = 2026").show(20, truncate=False)

StatementMeta(, 3692a400-3871-4b11-8a3a-6a0b28097f82, 13, Finished, Available, Finished, False)

+----------+------------------------------------+
|date      |holiday_name                        |
+----------+------------------------------------+
|2026-01-01|New Year's Day                      |
|2026-01-19|Martin Luther King Jr. Day          |
|2026-02-16|Washington's Birthday               |
|2026-05-25|Memorial Day                        |
|2026-06-19|Juneteenth National Independence Day|
|2026-07-03|Independence Day (observed)         |
|2026-07-04|Independence Day                    |
|2026-09-07|Labor Day                           |
|2026-10-12|Columbus Day                        |
|2026-11-11|Veterans Day                        |
|2026-11-26|Thanksgiving Day                    |
|2026-12-25|Christmas Day                       |
+----------+------------------------------------+



In [1]:
import datetime as dt
from pyspark.sql import functions as F, Row

fact   = spark.table("gold.fact_ridership_daily")
d_date = spark.table("gold.dim_date")
d_st   = spark.table("gold.dim_station")
d_fc   = spark.table("gold.dim_fare_class")
silver = spark.table("silver.ridership_hourly")

results = []
def check(name, passed, detail=""):
    results.append((name, "PASS" if passed else "FAIL", str(detail)))
def info(name, detail=""):
    results.append((name, "INFO", str(detail)))
def dup_count(df, cols):
    return df.groupBy(*cols).count().filter("count > 1").count()

# ---- 1. keys and grain ----
check("dim_date: date_key unique", dup_count(d_date, ["date_key"]) == 0)
check("dim_date: date unique", dup_count(d_date, ["date"]) == 0)
check("dim_station: station_key unique", dup_count(d_st, ["station_key"]) == 0)
check("dim_station: station_complex_id unique", dup_count(d_st, ["station_complex_id"]) == 0)
check("dim_fare_class: fare_class_key unique", dup_count(d_fc, ["fare_class_key"]) == 0)
check("dim_fare_class: fare_class_category unique", dup_count(d_fc, ["fare_class_category"]) == 0)
dups = dup_count(fact, ["date_key", "station_key", "fare_class_key"])
check("fact: one row per date x station x fare class", dups == 0, f"duplicate keys: {dups}")

# ---- 2. nulls and referential integrity ----
cols = ["date_key", "station_key", "fare_class_key", "ridership", "transfers", "hours_with_data"]
nulls = fact.select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).first().asDict()
check("fact: no nulls in keys or measures", sum(nulls.values()) == 0, nulls)
for dim, key in [(d_date, "date_key"), (d_st, "station_key"), (d_fc, "fare_class_key")]:
    orphans = fact.join(dim.select(key), key, "left_anti").count()
    check(f"fact: every {key} exists in its dimension", orphans == 0, f"orphan rows: {orphans}")

# ---- 3. calendar ----
r = d_date.agg(F.min("date").alias("mn"), F.max("date").alias("mx"), F.count("*").alias("n")).first()
expected = (r.mx - r.mn).days + 1
check("dim_date: continuous calendar, no gaps", r.n == expected, f"{r.mn} to {r.mx}, rows {r.n}, expected {expected}")
check("dim_date: is_weekend matches day_of_week",
      d_date.filter(F.col("is_weekend") != (F.col("day_of_week") >= 6)).count() == 0)
for name_col, num_col in [("month_name", "month"), ("day_name", "day_of_week")]:
    a = d_date.groupBy(name_col).agg(F.countDistinct(num_col).alias("c")).filter("c > 1").count()
    b = d_date.groupBy(num_col).agg(F.countDistinct(name_col).alias("c")).filter("c > 1").count()
    check(f"dim_date: {name_col} <-> {num_col} is 1:1 (sort-by-column valid)", a == 0 and b == 0)

rng = fact.agg(F.min("date_key").alias("a"), F.max("date_key").alias("b")).first()
missing = (d_date.filter((F.col("date_key") >= rng.a) & (F.col("date_key") <= rng.b)).select("date_key")
           .join(fact.select("date_key").distinct(), "date_key", "left_anti"))
n_missing = missing.count()
check("fact: no missing days between first and last date", n_missing == 0,
      [x.date_key for x in missing.limit(10).collect()] if n_missing else "")

# ---- 4. measure sanity ----
m = fact.agg(F.min("ridership").alias("min_r"),
             F.min("hours_with_data").alias("min_h"), F.max("hours_with_data").alias("max_h"),
             F.sum((F.col("transfers") < 0).cast("int")).alias("neg_tr"),
             F.sum((F.col("transfers") > F.col("ridership")).cast("int")).alias("tr_gt_r")).first()
check("fact: ridership above zero on every row", m.min_r > 0, f"min: {m.min_r}")
check("fact: hours_with_data between 1 and 24", m.min_h >= 1 and m.max_h <= 24, f"{m.min_h} to {m.max_h}")
check("fact: transfers never exceed ridership", m.tr_gt_r == 0, f"rows: {m.tr_gt_r}")
info("fact: rows with negative transfers (known source quirk)", m.neg_tr)

# ---- 5. Silver vs Gold reconciliation ----
s_tot = silver.agg(F.sum("ridership").alias("r"), F.sum("transfers").alias("t")).first()
f_tot = fact.agg(F.sum("ridership").alias("r"), F.sum("transfers").alias("t")).first()
check("Silver vs Gold: total ridership equal", s_tot.r == f_tot.r, f"{s_tot.r} vs {f_tot.r}")
check("Silver vs Gold: total transfers equal", s_tot.t == f_tot.t, f"{s_tot.t} vs {f_tot.t}")
s_y = silver.groupBy(F.col("ridership_year").alias("yr")).agg(F.sum("ridership").alias("s"))
f_y = (fact.join(d_date.select("date_key", "year"), "date_key")
           .groupBy(F.col("year").alias("yr")).agg(F.sum("ridership").alias("f")))
bad_years = s_y.join(f_y, "yr", "full_outer").filter("s is null or f is null or s != f").count()
check("Silver vs Gold: ridership equal for every year", bad_years == 0, f"mismatched years: {bad_years}")
check("Silver vs Gold: same number of stations",
      silver.select("station_complex_id").distinct().count() == d_st.count())

# ---- 6. dimension quality ----
st_cols = ["station_complex_id", "station_complex", "borough", "transit_mode", "latitude", "longitude"]
check("dim_station: no nulls in key attributes",
      d_st.filter(" or ".join(f"{c} is null" for c in st_cols)).count() == 0)
bad_geo = d_st.filter(~F.col("latitude").between(40.4, 41.0) | ~F.col("longitude").between(-74.3, -73.6)).count()
check("dim_station: coordinates inside the NYC area", bad_geo == 0, f"rows outside: {bad_geo}")
check("dim_station: station_complex names unique (hierarchy-safe)",
      d_st.select("station_complex").distinct().count() == d_st.count())
boroughs = {x[0] for x in d_st.select("borough").distinct().collect()}
check("dim_station: borough values valid",
      boroughs <= {"Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"}, sorted(boroughs))
check("dim_fare_class: 12 fare classes", d_fc.count() == 12, d_fc.count())
check("dim_fare_class: payment_method is OMNY or MetroCard",
      {x[0] for x in d_fc.select("payment_method").distinct().collect()} <= {"OMNY", "MetroCard"})
info("dim_station: stations with no fact rows", d_st.join(fact.select("station_key").distinct(), "station_key", "left_anti").count())
info("dim_fare_class: fare classes with no fact rows", d_fc.join(fact.select("fare_class_key").distinct(), "fare_class_key", "left_anti").count())

# ---- results ----
run_ts = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
res = spark.createDataFrame([Row(run_ts=run_ts, check_name=n, status=s, detail=d) for n, s, d in results])
res.write.format("delta").mode("append").saveAsTable("silver.dq_results")
res.select("status", "check_name", "detail").show(100, truncate=False)
print("FAILED:", res.filter("status = 'FAIL'").count())

StatementMeta(, a02fa9dd-becd-47fa-9fab-25e83a18ea8d, 3, Finished, Available, Finished, False)

+------+----------------------------------------------------------------+------------------------------------------------------------------------------------------------------------+
|status|check_name                                                      |detail                                                                                                      |
+------+----------------------------------------------------------------+------------------------------------------------------------------------------------------------------------+
|PASS  |dim_date: date_key unique                                       |                                                                                                            |
|PASS  |dim_date: date unique                                           |                                                                                                            |
|PASS  |dim_station: station_key unique                                 |            